# 진성 방식: 1차 READY 이후 KOSIS 검증

공통 문맥+중첩 방식으로 만든 `05_hcx_measurements.csv`에서 시작합니다. HCX를 다시 호출하지 않고 통계표 검색, 메타 조회, ITEM/OBJ 확정, 2차 READY, actual_value 비교와 verdict까지 실행합니다.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

import csv
import os
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/repro-baseline-20260727'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
SOURCE_RUN_DIR = DRIVE_ROOT / 'runs' / 'contextual_top50_context_v2_8x3'
INPUT_CSV = SOURCE_RUN_DIR / '05_hcx_measurements.csv'
OUT_DIR = SOURCE_RUN_DIR / '07_mapping_jinsung'
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('input :', INPUT_CSV)
print('output:', OUT_DIR)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'requests>=2.31,<3', 'python-dotenv>=1.0,<2',
    'numpy>=1.26,<3', 'sentence-transformers>=3.4,<6', 'transformers>=4.45,<6'
], check=True)
os.chdir(REPO_DIR)
print('commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
if not INPUT_CSV.exists():
    print('진성 방식의 05_hcx_measurements.csv를 선택하세요.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('05_hcx_measurements.csv 한 개만 업로드하세요.')
    INPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(next(iter(uploaded)), INPUT_CSV)

required = [
    (INPUT_CSV, 'HCX measurements'),
    (INDEX_DIR / 'manifest.json', 'BGE manifest'),
    (INDEX_DIR / 'embeddings.npy', 'BGE embeddings'),
    (REPO_DIR / 'kosis_table_summary.csv', 'KOSIS table index'),
]
for path, label in required:
    if not path.exists():
        raise FileNotFoundError(f'{label}가 없습니다: {path}')

if not os.environ.get('KOSIS_API_KEY'):
    os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY') or ''
if not os.environ.get('KOSIS_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')
print('입력·BGE 인덱스·KOSIS 키 확인 완료')

In [ ]:
command = [
    sys.executable, '-u', str(REPO_DIR / 'run_kosis_measurement_pipeline.py'),
    '--input', str(INPUT_CSV),
    '--table-index', str(REPO_DIR / 'kosis_table_summary.csv'),
    '--out-dir', str(OUT_DIR),
    '--retrieval-mode', 'hybrid',
    '--semantic-index', str(INDEX_DIR),
    '--semantic-top-k', '50',
    '--rerank-top-k', '20',
    '--top-tables', '5',
    '--item-top-k', '3',
    '--obj-top-k', '2',
    '--max-combinations', '20',
    '--device', 'cuda',
    '--delay', '0.12',
    '--verify',
]
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
def read_rows(path):
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

stem = INPUT_CSV.stem
paths = {
    '1차 READY': OUT_DIR / f'{stem}_kosis_ready.csv',
    '후보+메타': OUT_DIR / f'{stem}_kosis_candidates_with_meta.csv',
    '2차 READY 전체': OUT_DIR / f'{stem}_kosis_validated_mappings.csv',
    'actual verdict': OUT_DIR / f'{stem}_kosis_verified.csv',
}
for label, path in paths.items():
    rows = read_rows(path)
    print(f'{label:16s}: {len(rows):,} rows -> {path}')

validated = read_rows(paths['2차 READY 전체'])
verified = read_rows(paths['actual verdict'])
print('\n2차 mapping_status:', Counter(row.get('mapping_status', '') for row in validated))
print('2차 mapping_reason:', Counter(row.get('mapping_reason', '') for row in validated).most_common(12))
print('\nverdict:', Counter(row.get('verdict', '') for row in verified))
print('verdict_code:', Counter(row.get('verdict_code', '') for row in verified).most_common(12))

In [ ]:
# 후보 885행을 measurement 177건 단위로 축약한 진단 파일을 만듭니다.
status_priority = {
    'READY': 0,
    'NEEDS_CONFIRMATION': 1,
    'MAPPING_FAILED': 2,
    'API_ERROR': 3,
    'NOT_EVALUATED': 4,
}
grouped = {}
for row in validated:
    key = row.get('claim_measurement_id') or row.get('claim_id')
    grouped.setdefault(key, []).append(row)

diagnosis = []
for key, candidates in grouped.items():
    best = min(
        candidates,
        key=lambda row: (
            status_priority.get(row.get('mapping_status', ''), 9),
            int(row.get('candidate_rank') or 999),
        ),
    )
    reason = best.get('mapping_reason', '')
    if best.get('mapping_status') == 'READY':
        action = 'VERIFY_ACTUAL_VALUE'
    elif reason in {'upstream table candidate is not decisive rank-1 READY'} or reason.startswith('top candidates have small margin'):
        action = 'REVIEW_TABLE_RANKING'
    elif reason == 'UNIT_MISMATCH':
        action = 'REVIEW_UNIT_OR_TABLE'
    elif reason in {'EMPTY_RESPONSE', 'INVALID_COMBINATION'}:
        action = 'REVIEW_ITEM_OBJ_PERIOD'
    elif reason == 'PERIOD_MISSING':
        action = 'ENRICH_PERIOD'
    else:
        action = 'MANUAL_REVIEW'
    diagnosis.append({
        'claim_measurement_id': key,
        'claim_id': best.get('claim_id', ''),
        'claim_text': best.get('claim_text', ''),
        'indicator': best.get('indicator', ''),
        'value': best.get('value', ''),
        'unit': best.get('unit', ''),
        'period': best.get('period', ''),
        'best_tbl_id': best.get('tbl_id', ''),
        'best_tbl_name': best.get('tbl_name', ''),
        'best_candidate_rank': best.get('candidate_rank', ''),
        'measurement_mapping_status': best.get('mapping_status', ''),
        'measurement_mapping_reason': reason,
        'next_action': action,
    })

diagnosis_path = OUT_DIR / 'diagnosis_measurement_level.csv'
with diagnosis_path.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(diagnosis[0]))
    writer.writeheader()
    writer.writerows(diagnosis)
print('measurement-level status:', Counter(row['measurement_mapping_status'] for row in diagnosis))
print('next action:', Counter(row['next_action'] for row in diagnosis))
print('진단 파일:', diagnosis_path)

## 최종 확인 파일

- `*_kosis_validated_mappings.csv`: 2차 READY와 탈락 사유
- `*_kosis_verified.csv`: KOSIS actual_value와 일치·불일치·판단불가 verdict
- `diagnosis_measurement_level.csv`: 후보 5개를 measurement별 한 행으로 축약한 병목 진단